In [20]:
import os
import cv2
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Sequential 
from tensorflow.keras import layers, Model , regularizers ,  Input,models
from tensorflow.keras.layers import Conv2D , MaxPooling2D, Flatten, Dense, Dropout , Input, TimeDistributed, LSTM, Dropout , Bidirectional,Concatenate
from tensorflow.keras.applications import MobileNetV2 , ResNet50 , EfficientNetV2S ,EfficientNetV2B0 
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras import regularizers
from tensorflow.keras import mixed_precision
import tensorflow_addons as tfa
from glob import glob

In [41]:


video_root = 'UCF-101'
frame_root = 'UCF101_frames'
os.makedirs(frame_root, exist_ok=True)

def extract_frames(video_path, output_dir, desired_fps=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video: {video_path}")
        return

    # Get original FPS to help with frame skipping
    original_fps = cap.get(cv2.CAP_PROP_FPS)
    if original_fps == 0:
        original_fps = desired_fps
    skip_frames = max(1, int(original_fps / desired_fps))
    frame_index = 0
    saved_frame_index = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Save only every skip_frames frame
        if frame_index % skip_frames == 0:
            # Optionally convert frame to a tensor if needed:
            # tf_frame = tf.convert_to_tensor(frame)
            frame_filename = os.path.join(output_dir, f"frame_{saved_frame_index:05d}.jpg")
            cv2.imwrite(frame_filename, frame)
            saved_frame_index += 1

        frame_index += 1
    cap.release()

for class_name in os.listdir(video_root):
    class_path = os.path.join(video_root, class_name)
    if os.path.isdir(class_path):
        for video_name in os.listdir(class_path):
            if video_name.endswith('.avi'):
                video_path = os.path.join(class_path, video_name)
                video_basename = os.path.splitext(video_name)[0]
                output_dir = os.path.join(frame_root, class_name, video_basename)
                os.makedirs(output_dir, exist_ok=True)
                extract_frames(video_path, output_dir, desired_fps=30)

print("Frame extraction completed.")

KeyboardInterrupt: 

In [21]:
DATASET_DIR    = 'UCF101_frames'
CLASS_IND_PTH  = 'ucfTrainTestlist/classInd.txt'
TRAIN_SPLIT    = 'ucfTrainTestlist/trainlist01.txt'
TEST_SPLIT     = 'ucfTrainTestlist/testlist01.txt'
FRAME_SIZE     = (128, 128)
K              = 30        # number of frames per clip
BATCH_SIZE     = 4

SELECTED_CLASSES = [
    'ApplyEyeMakeup','ApplyLipstick','Archery','BabyCrawling','BalanceBeam',
    'BandMarching','Basketball','BasketballDunk','BenchPress','Biking',
    'Billiards','BlowDryHair','BlowingCandles','BodyWeightSquats','Bowling',
    'BoxingPunchingBag','BoxingSpeedBag','BreastStroke','BrushingTeeth','CleanAndJerk',
    'CliffDiving','CricketBowling','CricketShot','CuttingInKitchen','Diving',
    'Drumming','Fencing','FieldHockeyPenalty','FloorGymnastics','FrisbeeCatch',
    'FrontCrawl','GolfSwing','Haircut','Hammering','HandstandPushups',
    'HandstandWalking','HeadMassage','HighJump','HulaHoop','IceDancing',
    'JavelinThrow','JumpRope','JumpingJack','Kayaking','Knitting',
    'LongJump','Lunges','MilitaryParade','Mixing','MoppingFloor'
]
NUM_CLASSES = len(SELECTED_CLASSES)
selected_label2index = {c:i for i,c in enumerate(SELECTED_CLASSES)}


In [22]:
def parse_filtered_split(split_pth, base_dir, with_label=True):
    vids, labels = [], []
    with open(split_pth, 'r') as f:
        for line in f:
            rel, *_ = line.strip().split()
            cls = rel.split('/')[0]
            vid_folder = rel.split('.')[0]
            full_path = os.path.join(base_dir, vid_folder)
            if cls in SELECTED_CLASSES and os.path.isdir(full_path):
                vids.append(full_path)
                if with_label:
                    labels.append(selected_label2index[cls])
    return vids, labels

train_video_paths, train_labels = parse_filtered_split(TRAIN_SPLIT, DATASET_DIR)
test_video_paths,  test_labels  = parse_filtered_split(TEST_SPLIT,  DATASET_DIR)



In [23]:
def sample_topk_motion_window(folder_path, k=K):
    # ensure string → str
    if isinstance(folder_path, bytes):
        folder_path = folder_path.decode('utf-8')

    frame_files = sorted(glob(os.path.join(folder_path, '*.jpg')))
    T = len(frame_files)
    if T < k:
        frame_files += [frame_files[-1]] * (k - T)
        T = len(frame_files)

    # compute frame‑to‑frame diffs
    diffs = np.zeros(T, dtype=np.float32)
    prev = cv2.cvtColor(cv2.imread(frame_files[0]), cv2.COLOR_BGR2GRAY)
    for t in range(1, T):
        curr = cv2.cvtColor(cv2.imread(frame_files[t]), cv2.COLOR_BGR2GRAY)
        diffs[t] = np.sum(np.abs(curr.astype(np.float32) - prev.astype(np.float32)))
        prev = curr

    # sliding sum over window length k
    window_sums = np.convolve(diffs, np.ones(k, dtype=np.float32), mode='valid')
    best_start = int(np.argmax(window_sums))

    selected = frame_files[best_start:best_start + k]
    # return as numpy array of bytes (tf.numpy_function requires fixed-shape np.ndarray)
    return np.array(selected, dtype=np.string_)


In [24]:
def sample_uniform_frames(folder_path, k=K):
    # decode bytes→str
    if isinstance(folder_path, bytes):
        folder_path = folder_path.decode('utf-8')

    files = sorted(glob(os.path.join(folder_path, '*.jpg')))
    T = len(files)
    if T == 0:
        # no frames
        return np.zeros((k, *FRAME_SIZE, 3), dtype=np.float32)

    # pad if too few
    if T < k:
        files += [files[-1]] * (k - T)
        T = len(files)

    # pick k uniformly spaced indices
    idxs = np.linspace(0, T - 1, k, dtype=int)
    selected = [files[i] for i in idxs]

    # load & preprocess
    out = []
    for p in selected:
        img = cv2.imread(p)
        img = cv2.resize(img, FRAME_SIZE)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        out.append(img.astype(np.float32) / 255.0)
    return np.stack(out, axis=0)  # shape: (K, H, W, 3)


In [25]:
def process_path_train(path, label):
    # get the K filenames
    frames_paths = tf.numpy_function(sample_topk_motion_window, [path], tf.string)
    frames_paths.set_shape((K,))

    # load + augment each
    frames = []
    for i in range(K):
        img = tf.io.read_file(frames_paths[i])
        img = tf.image.decode_jpeg(img, channels=3)
        # spatial augmentations
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.2)
        img = tf.image.random_contrast(img, lower=0.7, upper=1.3)
        img = tf.image.resize(img, FRAME_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        frames.append(img)

    return tf.stack(frames, axis=0), label  # (K, H, W, 3), scalar

def process_path_test(path, label):
    frames_paths = tf.numpy_function(sample_topk_motion_window, [path], tf.string)
    frames_paths.set_shape((K,))

    frames = []
    for i in range(K):
        img = tf.io.read_file(frames_paths[i])
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, FRAME_SIZE)
        img = tf.cast(img, tf.float32) / 255.0
        frames.append(img)

    return tf.stack(frames, axis=0), label


In [26]:
train_ds = (
    tf.data.Dataset.from_tensor_slices((train_video_paths, train_labels))
    .shuffle(len(train_video_paths))
    .map(process_path_train, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((test_video_paths, test_labels))
    .map(process_path_test, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [27]:
for batch_frames, batch_labels in train_ds.take(1):
    print("Batch Frames Shape:", batch_frames.shape)  # (BATCH_SIZE, K, H, W, 3)
    print("Batch Labels:Shape", batch_labels.shape)


Batch Frames Shape: (4, 30, 128, 128, 3)
Batch Labels:Shape (4,)


In [35]:
def residual_block_3d(x, filters):
    shortcut = x
    
   
    x = layers.Conv3D(filters, (3, 3, 3), padding='same',
                      activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.SpatialDropout3D(0.2)(x)

   
    x = layers.Conv3D(filters, (3, 3, 3), padding='same',
                      activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

   
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv3D(filters, (1, 1, 1), padding='same',
                                 activation='relu',
                                 kernel_regularizer=regularizers.l2(1e-4))(shortcut)

    x = layers.Add()([shortcut, x])
    return x 

def build_3d_cnn_residual_model(
    input_shape=(K, FRAME_SIZE[0], FRAME_SIZE[1], 3),
    num_classes=NUM_CLASSES
):
    inputs = Input(shape=input_shape)

    # Initial Conv3D + Pool
    x = layers.Conv3D(32, (3, 3, 3), padding='same',
                      activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling3D((1, 2, 2))(x)

    # Residual Blocks
    x = residual_block_3d(x, 64)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = residual_block_3d(x, 128)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = residual_block_3d(x, 256)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    # Classification Head
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax',
                            kernel_regularizer=regularizers.l2(1e-4))(x)

    # Create model
    model = models.Model(inputs, outputs)

    # Compile with label smoothing loss
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-2),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

    return model

# Build model
model = build_3d_cnn_residual_model(
    input_shape=(K, FRAME_SIZE[0], FRAME_SIZE[1], 3),
    num_classes=NUM_CLASSES
)
model.summary()

Model: "model_8"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_9 (InputLayer)           [(None, 30, 128, 12  0           []                               
                                8, 3)]                                                            
                                                                                                  
 conv3d_80 (Conv3D)             (None, 30, 128, 128  2624        ['input_9[0][0]']                
                                , 32)                                                             
                                                                                                  
 batch_normalization_56 (BatchN  (None, 30, 128, 128  128        ['conv3d_80[0][0]']              
 ormalization)                  , 32)                                                       

In [41]:
def conv2plus1d_block(x, filters, kernel_size=(3, 3, 3), padding='same', strides=(1,1,1)):
    # First spatial conv
    x = layers.Conv3D(filters, (1, kernel_size[1], kernel_size[2]),
                      strides=(1, strides[1], strides[2]),
                      padding=padding,
                      activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)

    # Then temporal conv
    x = layers.Conv3D(filters, (kernel_size[0], 1, 1),
                      strides=(strides[0], 1, 1),
                      padding=padding,
                      activation='relu',
                      kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    return x

def build_cnn_2plus1d_model(
    input_shape=(30, 128, 128, 3), 
    num_classes=50
):
    inputs = Input(shape=input_shape)

    x = conv2plus1d_block(inputs, 32)
    x = layers.MaxPooling3D((1, 2, 2))(x)

    x = conv2plus1d_block(x, 64)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = conv2plus1d_block(x, 128)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = conv2plus1d_block(x, 256)
    x = layers.MaxPooling3D((2, 2, 2))(x)

    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )
    
    return model

# Instantiate model
model = build_cnn_2plus1d_model()
model.summary()


Model: "model_10"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_11 (InputLayer)       [(None, 30, 128, 128, 3)  0         
                             ]                                   
                                                                 
 conv3d_98 (Conv3D)          (None, 30, 128, 128, 32)  896       
                                                                 
 batch_normalization_71 (Bat  (None, 30, 128, 128, 32)  128      
 chNormalization)                                                
                                                                 
 conv3d_99 (Conv3D)          (None, 30, 128, 128, 32)  3104      
                                                                 
 batch_normalization_72 (Bat  (None, 30, 128, 128, 32)  128      
 chNormalization)                                                
                                                          

In [42]:
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    'best_model.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

earlystop_cb = tf.keras.callbacks.EarlyStopping(
    patience=5,
    restore_best_weights=True,
    monitor='val_accuracy',
    mode='max'
)

In [43]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=20,
    callbacks=[checkpoint_cb, earlystop_cb]
)


Epoch 1/20
1189/1189 [==============================] - 682s 569ms/step - loss: 4.3092 - accuracy: 0.0379 - val_loss: 3.6668 - val_accuracy: 0.1005
Epoch 2/20
1189/1189 [==============================] - 672s 564ms/step - loss: 3.8149 - accuracy: 0.0821 - val_loss: 3.4748 - val_accuracy: 0.1545
Epoch 3/20
1189/1189 [==============================] - 672s 565ms/step - loss: 3.6239 - accuracy: 0.1166 - val_loss: 3.2902 - val_accuracy: 0.1758
Epoch 4/20
1189/1189 [==============================] - 699s 587ms/step - loss: 3.4519 - accuracy: 0.1496 - val_loss: 3.2479 - val_accuracy: 0.1646
Epoch 5/20
1189/1189 [==============================] - 700s 588ms/step - loss: 3.2927 - accuracy: 0.1740 - val_loss: 3.0730 - val_accuracy: 0.1860
Epoch 6/20
1189/1189 [==============================] - 699s 587ms/step - loss: 3.1436 - accuracy: 0.1973 - val_loss: 3.1509 - val_accuracy: 0.1940
Epoch 7/20
1189/1189 [==============================] - 702s 590ms/step - loss: 3.0201 - accuracy: 0.2140 - val_